# chunk_v2_run — BM25 mức đoạn + gộp đoạn 1800 ký tự, đo trên dev300

**~3-5h.** GPU T4, Internet On. Chạy **interactive** hoặc commit đều được (dư giờ so với trần 12h).

Đổi hai thứ trong `deep_chunk.py`, **cả hai đều không tốn thêm GPU** (chạy ở CPU lúc băm):

1. `pick_chunks` bỏ `len(qs & terms)` → **BM25 trong phạm vi văn bản**
2. `MERGE_CHARS = 1800` gộp đoạn liền kề (`chunks_of` cắt theo Điều nên đoạn chỉ ~650 ký tự)

Đã đo trước trên `oracle_chunks_dev300.json`, **0 giờ GPU**, và `deep_chunk.py` mới
tái lập đúng hai số này trên CPU:

| hàm chọn | `chunks_of` gốc | gộp 1800 |
|---|---|---|
| `count` (cũ) | 0.9083 ← mốc | 0.9183 |
| **`bm25` mức đoạn** | **0.9217** | **0.9250** ← cấu hình lượt này |
| *chọn HOÀN HẢO (trần)* | *0.9250* | *0.9317* |

**Hai biến một lượt là CỐ Ý, không phải phạm quy tắc 3.** Quy tắc 3 tồn tại để quy công
khi thắng — mà oracle đã quy công sẵn trên CPU: BM25 ăn +1,34, gộp đoạn thêm +0,33.
Tách ra chạy hai lượt 5h để đo hai hiệu ứng đều dưới ngưỡng phân giải của dev300 là
đốt quota vô ích.

**Việc thật của lượt này là LOẠI TRỪ TỤT ĐIỂM, không phải đo mức tăng.** +1,67 nằm dưới
ngưỡng 2,0 nên dev300 không phân giải nổi — mức tăng thật sẽ đo bằng public LB (thứ vừa
phân giải +1,27 mà dev300 mù). Nhưng gộp đoạn đổi đầu vào cross-encoder cho **cả 20 văn
bản**, nên có rủi ro pha loãng làm TỆ ĐI. Tụt ≥2 điểm thì dev300 bắt được thừa sức.

| kết quả `max n=2` | quyết định |
|---|---|
| ≥ 0.9083 | không tụt → chạy đề thi, nộp, để LB đo mức tăng |
| 0.89 – 0.9083 | pha loãng ăn mất phần thắng → thử `MERGE_CHARS=0`, giữ mỗi BM25 |
| < 0.89 | tụt thật → quay về `deep_chunk.py` cũ |

**Upload lên dataset `project-ir` — đúng MỘT file:** `deep_chunk.py` (bản mới, ĐÈ bản cũ).
`scores_dev300_deep_M20_K20.json` đã upload từ lượt oracle.

In [ ]:
!pip install -q sentence-transformers

import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
INPUT_DIR = "/kaggle/input/project-ir"     # đổi đúng slug — xem output bên dưới
sys.path.append(INPUT_DIR)
!ls /kaggle/input

In [ ]:
import json, time
from pathlib import Path

from rerank import load_reranker
from rerank_from_d import blend_bm25_first
import deep_chunk as DC

DEV_GOLD = f"{INPUT_DIR}/dev_300_locked.json"
SCORES   = f"{INPUT_DIR}/scores_dev300_deep_M20_K20.json"
CTX_DIR  = f"{INPUT_DIR}/selected-contexts"
M_DOC, K_CHUNK, TOPK = 20, 20, 5
BASELINE = 0.8883        # base n=2, tầng 1 — KHÔNG được đổi, lượt này không đụng tầng 1
MOC_CU   = 0.9083        # max n=2 của deep_chunk cũ — mốc phải vượt
OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DC.MERGE_CHARS = 1800                      # <-- CẦN GẠT 1. Đặt TRƯỚC mọi lời gọi (nó đổi cache)
assert hasattr(DC, "MERGE_CHARS"), "deep_chunk.py trên dataset vẫn là BẢN CŨ — upload lại"
assert os.path.isdir(CTX_DIR), f"KHÔNG thấy {CTX_DIR}"
assert os.path.isfile(SCORES), f"CHƯA UPLOAD {SCORES}"

## Bước 1 — Tầng 1 lấy từ file, KHÔNG chạy lại (tiết kiệm ~30 phút)

Lượt này không đụng tầng 1 (tầng 1 dùng 3 đoạn của D, gộp đoạn chỉ ảnh hưởng tầng 2).

**Bóc `ce_deep` cũ ra.** Nó sinh từ cách băm khác — giữ lại là trộn hai hệ đo,
`max` sẽ lấy nhầm điểm của đoạn cũ. `ce` và `bm25` giữ nguyên.

In [ ]:
dev = json.load(open(DEV_GOLD, encoding="utf-8"))
old = json.load(open(SCORES,   encoding="utf-8"))
dev_q    = {q: v["question"] for q, v in dev.items()}
gold     = {q: {str(x) for x in v["answer"]} for q, v in dev.items()}
scores_base = {q: {d: {"ce": v["ce"], "bm25": v["bm25"]} for d, v in s.items()}
               for q, s in old.items()}                       # <- ce_deep cũ bị bóc ở đây
bm25 = {q: sorted(scores_base[q], key=lambda d: -scores_base[q][d]["bm25"]) for q in dev_q}

def rec(S, variant, n):
    pred = {q: blend_bm25_first(DC.rank_by(S[q], variant), bm25[q], k=TOPK, n_bm25=n)
            for q in dev_q}
    return sum(len(gold[q] & set(pred[q])) / len(gold[q]) for q in gold) / len(gold)

chk = rec(scores_base, "base", 2)
print(f"base n=2 = {chk:.4f}  (phải là {BASELINE})"
      + ("  OK" if abs(chk - BASELINE) < 0.002 else "  <-- LỆCH, DỪNG LẠI"))
assert not any("ce_deep" in v for s in scores_base.values() for v in s.values()), "ce_deep cũ chưa bóc sạch"
print(f"{len(dev)} câu | corpus {len(os.listdir(CTX_DIR)):,} file")

## Bước 2 — ĐẾM trước khi chấm (quy tắc 6)

Bản cũ ở M=20/K=20 là **~118.000 đoạn**. Gộp đoạn làm số đoạn/văn bản giảm ~2,4x nên
con số này nên **thấp hơn hoặc xấp xỉ**, nhưng mỗi đoạn dài gấp 2,2x nên nhịp đoạn/s có
thể chậm đi. Đó là lý do Bước 4 in nhịp thật sau 25 câu đầu — xem rồi hẵng để nó chạy.

In [ ]:
t0 = time.time()
n2 = DC.count_deep_chunks(dev_q, scores_base, CTX_DIR, M_DOC, K_CHUNK)
print(f"tầng 2 sẽ chấm {n2:,} đoạn ({n2/len(dev_q):.0f}/câu) | băm+đếm {time.time()-t0:.0f}s")
print(f"ước {n2/12.9/3600:.1f}h ở nhịp cũ 12,9 đoạn/s — chậm đi 2x thì {n2/6.5/3600:.1f}h")

sizes = [len(DC.doc_chunks(CTX_DIR, d)) for q in list(dev_q)[:60] for d in list(scores_base[q])[:5]]
sizes.sort()
print(f"đoạn/văn bản sau khi gộp: trung vị {sizes[len(sizes)//2]}, max {sizes[-1]}"
      f" | {sum(s <= K_CHUNK for s in sizes)/len(sizes):.0%} văn bản có ≤{K_CHUNK} đoạn (lấy hết, khỏi chọn)")
assert n2 < 200_000, "quá nhiều — kiểm lại M_DOC / K_CHUNK / MERGE_CHARS"

## Bước 3 — Load reranker (GỐC, không phải ft_model)

In [ ]:
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda")

## Bước 4 — Tầng 2

Dòng tiến độ in mỗi 25 câu. **Xem dòng đầu tiên rồi mới bỏ đi làm việc khác:** nếu ETA
vượt 6h thì dừng, hạ `K_CHUNK` xuống 12 (gộp đoạn rồi thì K=12 vẫn phủ hơn K=20 bản cũ).

In [ ]:
t0 = time.time()
scores_deep = DC.deepen_all(dev_q, scores_base, CTX_DIR, score_fn, M_DOC, K_CHUNK)
el = time.time() - t0
print(f"tầng 2 xong trong {el/60:.0f} phút | nhịp thật {n2/el:.1f} đoạn/s (cũ: 12,9)")

p = f"{OUTPUT_DIR}/scores_dev300_bm25pick_merge1800_M20_K20.json"
json.dump(scores_deep, open(p, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p} — TẢI VỀ dù phần dưới có hỏng (quy tắc 2)")

## Bước 5 — Đo: 3 biến thể × 4 giá trị `n_bm25`

`base` phải vẫn ra **0.8883** — lượt này không đụng tầng 1, lệch là có gì sai.

`n_bm25` quét lại: điểm đổi thì đỉnh có thể dời. Nhưng **nếu đỉnh dời khỏi n=2 thì đừng
vội đổi** — n=2 đã được xác nhận bằng public LB thật (0.8573 vs 0.8532/0.8350/0.8424),
còn dev300 đã một lần chọn nhầm n=1 hôm 11/08.

In [ ]:
tab = {v: {n: rec(scores_deep, v, n) for n in (0, 1, 2, 3)} for v in DC.VARIANTS}
print(f"{'biến thể':10s}" + "".join(f"  n={n}    " for n in (0, 1, 2, 3)))
for v, row in tab.items():
    print(f"{v:10s}" + "".join(f"  {row[n]:.4f} " for n in (0, 1, 2, 3)))

got = tab["max"][2]
print(f"\nbase n=2  = {tab['base'][2]:.4f}  (phải {BASELINE})")
print(f"max  n=2  = {got:.4f}  | mốc cũ {MOC_CU}  | Δ {(got-MOC_CU)*100:+.2f} điểm")
print(f"dự đoán offline từ oracle: 0.9250 (Δ +1,67)")
print("\n" + "=" * 66)
if got >= MOC_CU:
    print("KHÔNG TỤT -> chạy đề thi (tầng 2, ~9h, có scores_public.json rồi), nộp,")
    print("để public LB đo mức tăng thật. dev300 không phân giải nổi +1,67.")
elif got >= 0.89:
    print("TỤT NHẸ -> pha loãng ăn mất phần thắng. Chạy lại với DC.MERGE_CHARS = 0,")
    print("giữ mỗi BM25 chọn đoạn (oracle nói riêng nó đã +1,34).")
else:
    print(f"TỤT THẬT ({(got-MOC_CU)*100:+.2f}) -> quay về deep_chunk.py cũ. Đừng cố cứu.")
print("=" * 66)

json.dump({"table": tab, "baseline": BASELINE, "moc_cu": MOC_CU, "max_n2": got,
           "delta": got - MOC_CU, "MERGE_CHARS": DC.MERGE_CHARS,
           "M_DOC": M_DOC, "K_CHUNK": K_CHUNK, "n_chunk_tang2": n2,
           "doan_moi_giay": n2 / el},
          open(f"{OUTPUT_DIR}/chunk_v2_eval.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}  {os.path.getsize(os.path.join(OUTPUT_DIR, f)):,} bytes")
print("TẢI TOÀN BỘ outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN")